
# Booking.com-Style Hotel Booking Funnel A/B Testing & Revenue Uplift Analysis

This notebook performs end-to-end analysis of a simulated A/B test for a hotel booking platform checkout funnel. We evaluate whether a redesigned checkout experience (Variant B) improved user conversion, revenue per visitor, and other key metrics relative to the existing checkout (Variant A). The dataset is synthetic and generated for this portfolio project.

*Experiment period:* April 25 – May 15, 2026 (21 days)
*Primary KPI:* Booking conversion rate
*Secondary KPIs:* Revenue per visitor, average booking value, checkout abandonment, payment success, and device-level performance.



## Load Data

In [ ]:

import pandas as pd
import numpy as np
from scipy import stats

# Load CSVs
users = pd.read_csv('../data/users.csv')
sessions = pd.read_csv('../data/sessions.csv')
assignments = pd.read_csv('../data/experiment_assignment.csv')
events = pd.read_csv('../data/funnel_events.csv', parse_dates=['event_time'])
bookings = pd.read_csv('../data/bookings.csv')

# Ensure date parsing
sessions['session_date'] = pd.to_datetime(sessions['session_date'])
assignments['assigned_at'] = pd.to_datetime(assignments['assigned_at'])
bookings['booking_date'] = pd.to_datetime(bookings['booking_date'])

print("Users:", users.shape, "Sessions:", sessions.shape, "Bookings:", bookings.shape)


## Overall Conversion Rates

In [ ]:

# Determine sessions per variant
sessions_variants = assignments.groupby('variant')['session_id'].nunique()

# Determine bookings per variant
bookings_variants = bookings.groupby('variant')['session_id'].nunique()

# Conversion rate per variant
conversion_rates = (bookings_variants / sessions_variants).rename('conversion_rate')

summary = pd.DataFrame({
    'sessions': sessions_variants,
    'bookings': bookings_variants,
    'conversion_rate': conversion_rates
})

print(summary)


## Statistical Significance

In [ ]:

# Two-proportion z-test between Variant A and Variant B
count = bookings_variants.values
nobs = sessions_variants.values

stat, pval = stats.proportions_ztest(count, nobs)

# Compute difference in conversion rate
diff = conversion_rates['B'] - conversion_rates['A']

# Compute standard error and 95% confidence interval
prop_pool = (count[0] + count[1]) / (nobs[0] + nobs[1])
std_error = np.sqrt(prop_pool * (1 - prop_pool) * (1/nobs[0] + 1/nobs[1]))
ci_low = diff - 1.96 * std_error
ci_high = diff + 1.96 * std_error

print(f"Z-statistic: {stat:.3f}
P-value: {pval:.4f}
Absolute uplift: {diff*100:.2f}%")
print(f"95% CI of uplift: ({ci_low*100:.2f}%, {ci_high*100:.2f}%)")


## Revenue Metrics

In [ ]:

# Total revenue and revenue per visitor per variant
revenue_variant = bookings.groupby('variant')['booking_value'].sum()
revenue_per_visitor = revenue_variant / sessions_variants
avg_booking_value = bookings.groupby('variant')['booking_value'].mean()

revenue_summary = pd.DataFrame({
    'total_revenue': revenue_variant,
    'revenue_per_visitor': revenue_per_visitor,
    'average_booking_value': avg_booking_value
})

print(revenue_summary)

# Incremental revenue
inc_revenue_per_visitor = revenue_per_visitor['B'] - revenue_per_visitor['A']
print(f"Incremental revenue per visitor (B vs A): {inc_revenue_per_visitor:.2f}")


## Segment-Level Uplift

In [ ]:

# Example: device-level conversion rates and uplift
segment = 'device'
segment_results = []
for variant in ['A','B']:
    # sessions by segment and variant
    sessions_seg = sessions.merge(assignments[['session_id','variant']], on='session_id')
    sessions_seg = sessions_seg[sessions_seg['variant'] == variant]
    # bookings by segment and variant
    bookings_seg = bookings[bookings['variant'] == variant]
    sessions_counts = sessions_seg.groupby(segment)['session_id'].nunique()
    bookings_counts = bookings_seg.groupby(segment)['session_id'].nunique()
    conv_rate = (bookings_counts / sessions_counts).rename('conversion_rate')
    segment_results.append(conv_rate)

segment_df = pd.concat(segment_results, axis=1, keys=['A','B'])
segment_df['absolute_uplift'] = segment_df['B'] - segment_df['A']
segment_df['relative_uplift'] = (segment_df['absolute_uplift'] / segment_df['A'])

print(segment_df)
